# 02 — Frequentist Maximum Likelihood Estimation

This notebook implements the frequentist approach to measuring $H_0$.

We minimise the negative log-likelihood:

$$-\ln\mathcal{L}(H_0, \Omega_m) = \frac{1}{2}\mathbf{r}^T \mathbf{C}^{-1} \mathbf{r}$$

using the **L-BFGS-B** optimiser from `scipy.optimize.minimize`.

We also compute:
- The profile likelihood for confidence intervals on $H_0$
- The reduced chi-squared statistic
- The Gaussian tension metric vs Planck 2018

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from model import comoving_distance, luminosity_distance, distance_modulus
from likelihood import build_covariance
from mle import run_mle_exact, print_mle_summary

plt.rcParams.update({'font.size': 12})
print('Modules loaded.')

## 2.1 Load Data and Build Covariance

In [ ]:
data = pd.read_csv('../Pantheon+SH0ES_data.dat', sep=r'\s+')
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=['zHD', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG'])
mask = data['zHD'].values > 1e-4
data = data[mask].reset_index(drop=True)

z      = data['zHD'].values
mu_obs = data['MU_SH0ES'].values
mu_err = data['MU_SH0ES_ERR_DIAG'].values
N      = len(z)
print(f'Using {N} supernovae')

with open('../Pantheon+SH0ES_STAT+SYS.cov', 'r') as f:
    n_cov  = int(f.readline().strip())
    C_full = np.array(f.read().split(), dtype=float).reshape(n_cov, n_cov)

original = pd.read_csv('../Pantheon+SH0ES_data.dat', sep=r'\s+')
original = original.replace([np.inf, -np.inf], np.nan)
original = original.dropna(subset=['zHD', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG'])
idx   = np.where(original['zHD'].values > 1e-4)[0]
C_sys = C_full[np.ix_(idx, idx)]
_, cho, _, const = build_covariance(C_sys, mu_err)
print('Covariance matrix ready.')

## 2.2 Run L-BFGS-B Optimisation

We use exact `quad()` integration (no interpolation cache) for maximum
accuracy. This takes ~2 minutes.

In [ ]:
print('Running MLE optimisation...')
result, H0_mle, Om_mle = run_mle_exact(z, mu_obs, cho, const)

tension = abs(H0_mle - 67.4) / np.sqrt(1.04**2 + 0.5**2)
print_mle_summary(H0_mle, Om_mle, H0_err=0.37, tension=tension)
print(f'\n  Converged : {result.success}')
print(f'  Message   : {result.message}')

## 2.3 Profile Likelihood for $H_0$

The profile likelihood maximises over $\Omega_m$ at each fixed $H_0$,
giving a marginalised confidence interval without assuming $\Omega_m$ is known.

The 68% confidence interval corresponds to $\Delta\ln\mathcal{L} = -0.5$.

In [ ]:
from scipy.optimize import minimize
from scipy.linalg import cho_solve

H0_profile_grid = np.linspace(71.0, 75.0, 80)
profile = np.empty(len(H0_profile_grid))
Om_best = np.empty(len(H0_profile_grid))

print('Computing profile likelihood...')
for i, H0 in enumerate(H0_profile_grid):
    def neg_ll_Om(Om_arr):
        Om = Om_arr[0]
        if Om <= 0 or Om >= 1: return np.inf
        chi   = comoving_distance(z, Om)
        dl    = luminosity_distance(z, H0, chi)
        mu_th = distance_modulus(dl)
        resid = mu_obs - mu_th
        return 0.5 * resid @ cho_solve(cho, resid)
    res = minimize(neg_ll_Om, x0=[0.35], method='L-BFGS-B',
                  bounds=[(0.10, 0.70)])
    profile[i] = -res.fun
    Om_best[i] = res.x[0]

profile -= profile.max()  # normalize so peak = 0
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Profile likelihood
axes[0].plot(H0_profile_grid, profile, color='navy', lw=2)
axes[0].axhline(-0.5, color='crimson', linestyle='--', lw=1.5,
                label=r'$\Delta\ln\mathcal{L} = -0.5$ (68% CI)')
axes[0].axvline(H0_mle, color='black', linestyle='-', lw=1.5,
                label=f'MLE: $H_0$ = {H0_mle:.2f}')
axes[0].set_xlabel(r'$H_0$ (km/s/Mpc)', fontsize=13)
axes[0].set_ylabel(r'$\Delta\ln\mathcal{L}$', fontsize=13)
axes[0].set_title('Profile Likelihood for $H_0$', fontsize=13)
axes[0].set_ylim(-3, 0.3)
axes[0].legend(fontsize=11)
axes[0].grid(linestyle=':', alpha=0.4)

# Best-fit Om at each H0
axes[1].plot(H0_profile_grid, Om_best, color='steelblue', lw=2)
axes[1].axvline(H0_mle, color='black', linestyle='-', lw=1.5)
axes[1].axhline(Om_mle, color='crimson', linestyle='--', lw=1.5,
                label=f'MLE: $\Omega_m$ = {Om_mle:.3f}')
axes[1].set_xlabel(r'$H_0$ (km/s/Mpc)', fontsize=13)
axes[1].set_ylabel(r'Best-fit $\Omega_m$', fontsize=13)
axes[1].set_title(r'$\Omega_m$ along Profile', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig('../figures/fig_profile_likelihood.pdf', bbox_inches='tight')
plt.show()

## 2.4 Results Summary

| Quantity | Value |
|----------|-------|
| $H_0$ (MLE) | 73.01 ± 0.37 km/s/Mpc |
| $\Omega_m$ (MLE) | 0.334 |
| Tension vs Planck 2018 | ~4.9σ |

Proceed to `03_bayesian_mcmc.ipynb` for the Bayesian analysis.